# 10. 멀티모달 임베딩 — OCR 없이 스캔 페이지 찾기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/10_multimodal_embeddings.ipynb)

> 🧭 **스토리** — 총무팀 캐비닛에서 나온 규정집은 **스캔 이미지**뿐이고, 직원들은 영수증 사진과 차트 캡처도 함께 올린다. 06~08번처럼 글자를 뽑아내지 않고, **이미지를 그대로 벡터로 바꿔** 한국어 질문으로 찾을 수 있을까?
>
> 📖 **이론 연결** — 이론2 p71(멀티모달 임베딩: 이미지·텍스트 한 공간, 문서 특화 모델, EG2) · ⏱ **15분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
09번과 같은 `sentence-transformers` 6.1 이상을 씁니다. 이번에는 EmbeddingGemma 2(EG2)의 **비전 인코더까지** 올립니다.
GPU가 없으면 이미지 수를 줄여(스캔 3쪽 + 이미지 2장) 진행할 수 있습니다. 환경변수 `NB_SMOKE=1`이면 자동으로 줄입니다.

In [ ]:
%pip -q install -U sentence-transformers transformers pillow

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-ai-api/data)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-ai-api/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ 검색 대상 이미지 8장 — 스캔 페이지 6장 + 페이지가 아닌 이미지 2장(영수증 사진, 막대그래프)
import os
from PIL import Image
import matplotlib.pyplot as plt

SMOKE = os.environ.get("NB_SMOKE") == "1"           # CPU에서 빨리 확인할 때: 스캔 3쪽만
pages = ["p2", "p4", "p5"] if SMOKE else ["p1", "p2", "p3", "p4", "p5", "p6"]
img_files = {p: f"scan/{p}.png" for p in pages}
img_files.update({"receipt": "images/receipt.png", "chart": "images/chart_allowance.png"})
images = {k: Image.open(fetch(v)).convert("RGB") for k, v in img_files.items()}

fig, axes = plt.subplots(1, len(images), figsize=(2.1 * len(images), 3))
for ax, (k, im) in zip(axes, images.items()):
    ax.imshow(im); ax.set_title(f"{k} {im.size[0]}×{im.size[1]}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

스캔 페이지는 약간 기울고 잡음이 섞여 있습니다(06번에서 본 그 파일). p2에는 국내 출장비 표, p3에는 해외 출장비 표, p4에는 연차·경조사 휴가 표, p5에는 경조금·교육비 표가 있습니다.

## 1. EG2 비전 포함 로드 🔵
09번에서는 `config_kwargs`로 비전·음성 부품을 모두 뺐습니다. 이번에는 **음성만 빼고** 비전은 켭니다(카드의 "Text and image" 설정, 약 440M).
float16은 금지입니다(값이 NaN이 되거나 조용히 망가짐). `torch.cuda.is_bf16_supported()`가 True면 bf16, 아니면(CPU) float32를 씁니다. 코랩 T4도 True를 돌려줘 bf16으로 올라갑니다. T4에는 bf16 전용 연산 장치가 없어 에뮬레이션으로 계산합니다.

In [ ]:
import time, torch, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

EG_DTYPE = torch.bfloat16 if (HAS_GPU and torch.cuda.is_bf16_supported()) else torch.float32   # 코랩 T4도 True → bf16(에뮬레이션)
t0 = time.time()
eg = SentenceTransformer("google/embeddinggemma-2", device=DEVICE,
                         model_kwargs={"dtype": EG_DTYPE},
                         config_kwargs={"audio_config": None})     # 텍스트+이미지. 전부 쓰려면 config_kwargs를 빼면 됩니다(740M)
n_params = sum(p.numel() for p in eg.parameters())
print(f"로드 {time.time()-t0:.1f}초 · dtype {EG_DTYPE} · 파라미터 {n_params/1e6:.0f}M · 차원 {eg.get_embedding_dimension()}")

💡 **해석** — 강사 환경에서 파라미터 439M이 찍혔습니다. 텍스트 전용(09번, 271M)보다 비전 인코더 약 170M이 늘었습니다. float32 기준 메모리는 약 1.8GB입니다.

## 2. 이미지를 벡터로 🔵
이미지는 `{"image": PIL이미지}` 딕셔너리로 넣습니다. **이미지에는 접두어(prompt_name)를 붙이지 않습니다.**
메모리를 아끼려고 한 장씩 넣습니다. 이미지 한 장은 기본 **280개의 시각 토큰(soft token)** 으로 요약된 뒤, 숫자 768개짜리 벡터 하나가 됩니다.

In [ ]:
img_names = list(images)
img_vecs, rows = [], []
for k in img_names:
    t0 = time.time()
    v = eg.encode({"image": images[k]}, normalize_embeddings=True)      # (768,)
    img_vecs.append(v); rows.append({"이미지": k, "초": round(time.time() - t0, 2)})
IMG = np.stack(img_vecs)
print("이미지 행렬:", IMG.shape, "← 이미지 8장 × 숫자 768개")
display(pd.DataFrame(rows).set_index("이미지").T)

💡 **해석** — 코랩 T4(bfloat16) 실측(세 번 실행)에서 첫 장은 2.2~2.5초(준비 비용), 나머지는 한 장에 약 0.7초였습니다. 강사 노트북(CPU, 다른 작업이 없을 때)은 첫 장 약 2초, 나머지 한 장에 약 0.3초였고, CPU가 바쁘면 몇 배씩 늘어났습니다. T4에는 bfloat16 전용 연산 장치가 없어(에뮬레이션) GPU인데도 생각보다 빠르지 않을 수 있습니다.
결과 `IMG`는 8행 × 768열 행렬입니다. **이미지 한 장이 숫자 768개 한 줄**이 됩니다. 그 숫자가 어떻게 생겼는지 2-1에서 봅니다.

### 2-1. 바뀐 벡터 들여다보기 — 이미지도 글도 숫자 768개
아래 셀은 이미지 3장(p2 표 페이지, 영수증 사진, 막대그래프)과 질문 글 2개를 같은 모델로 바꾼 결과를 나란히 보여 줍니다.
- 왼쪽: 원래 입력(이미지 또는 글)
- 오른쪽: 바뀐 벡터. 768칸짜리 띠 한 줄이고, 칸 하나가 숫자 하나입니다(빨강 = 양수, 파랑 = 음수, 진할수록 절댓값이 큼).

글은 09번처럼 질문용 `prompt_name="SearchQuery"`를 붙여 바꿉니다. 마지막 표는 이 벡터끼리 잰 코사인 유사도입니다.

In [ ]:
# 🔵 이미지 3장과 질문 글 2개가 각각 어떤 숫자 768개로 바뀌었나
demo_q = ["호텔 영수증", "과장 국내 출장 일비 표"]
Q_DEMO = eg.encode(demo_q, prompt_name="SearchQuery", normalize_embeddings=True)
demo = [("이미지", k, IMG[img_names.index(k)]) for k in ["p2", "receipt", "chart"] if k in img_names]
demo += [("글", q, v) for q, v in zip(demo_q, Q_DEMO)]

lim = float(np.abs(np.stack([v for _, _, v in demo])).max())
fig, axes = plt.subplots(len(demo), 2, figsize=(12, 1.9 * len(demo)), gridspec_kw={"width_ratios": [1.5, 7]})
for (kind, name, v), (a0, a1) in zip(demo, axes):
    if kind == "이미지":
        a0.imshow(images[name])
    else:
        a0.text(0.5, 0.5, f"'{name}'", ha="center", va="center", fontsize=9, wrap=True)
    a0.axis("off")
    a1.imshow(v[None, :], aspect="auto", cmap="coolwarm", vmin=-lim, vmax=lim)    # 768칸 띠
    a1.set_yticks([]); a1.set_xticks([0, 255, 511, 767], ["1", "256", "512", "768"])
    a1.set_title(f"[{kind}] {name} → 숫자 768개 · 앞 5개: " + "  ".join(f"{x:.3f}" for x in v[:5]), fontsize=9, loc="left")
plt.tight_layout(); plt.show()

display(pd.DataFrame([{"입력": name, "종류": kind, "벡터 크기": v.shape[0], "길이(노름)": round(float(np.linalg.norm(v)), 3),
                       "앞 5개 값": [round(float(x), 3) for x in v[:5]]} for kind, name, v in demo]).set_index("입력"))

# 같은 공간의 벡터이므로 글 벡터와 이미지 벡터를 바로 비교할 수 있다 (길이가 1이라 내적 = 코사인 유사도)
D_IMG = np.stack([v for kind, _, v in demo if kind == "이미지"])
cos = pd.DataFrame(Q_DEMO @ D_IMG.T, index=[f"글 '{q}'" for q in demo_q],
                   columns=[f"이미지 {n}" for kind, n, _ in demo if kind == "이미지"]).round(3)
print("글 벡터 × 이미지 벡터 코사인 유사도 (클수록 가깝다)")
display(cos)

💡 **해석** — 코랩 T4 실측입니다.
- 다섯 입력 모두 **숫자 768개짜리 벡터**가 되었고, 길이(노름)는 약 1입니다(`normalize_embeddings=True`. T4에서는 bfloat16으로 계산해 0.999~1.002). 이미지든 글이든 모양이 같아서 서로 비교할 수 있습니다.
- 다섯 띠가 비슷해 보이는 것은 몇몇 칸(세로로 진한 줄)이 다섯 입력 모두에서 같은 부호로 크기 때문입니다. 칸 하나하나에는 사람이 읽을 뜻이 없고, 의미는 768개를 한꺼번에 비교한 **코사인 유사도**에서 나옵니다.
- 마지막 표에서 글 '호텔 영수증'은 영수증 이미지와 0.781로 가장 가깝습니다(p2 0.617, chart 0.641).
- 글 '과장 국내 출장 일비 표'는 정답인 p2(0.709)보다 막대그래프 chart(0.761)와 더 가깝습니다. chart는 직급별 국내 출장비(일비·숙박비)를 그린 그래프라 내용이 겹칩니다. 이 차이가 3절 검색 결과로 그대로 이어집니다.
- 같은 쌍인데 3절 표에서는 셋째 자리가 조금 다를 수 있습니다(예: 영수증 0.781 → 0.779). bfloat16 계산에서 한 번에 넣는 질문 묶음이 달라 끝자리가 흔들린 것으로 보이며, 순위에는 영향이 없었습니다.

## 3. 한국어 질문으로 이미지 찾기 🔵
검색은 2-1의 마지막 표를 **질문 8개 × 이미지 8장**으로 넓힌 것입니다.
1. 질문 글을 벡터로 바꾼다(`prompt_name="SearchQuery"`).
2. 이미지 8장의 벡터와 하나씩 코사인 유사도를 잰다 → 점수 8개.
3. 점수가 높은 순서로 줄 세운다. **1등이 미리 정해 둔 정답 이미지면 '맞음'** 입니다.

먼저 질문 하나로 이 과정을 그림으로 보고(3-1), 질문 8개 전체를 그림으로 본 뒤(3-2), 마지막에 표로 정리합니다(3-3).

### 3-1. 질문 하나로 보기 — '연차휴가'
아래 셀은 질문 8개와 정답 이미지 목록, 검색 함수를 만든 뒤, 질문 '연차휴가'의 점수를 이미지 8장 모두에 대해 그립니다.

In [ ]:
queries = {   # 질문: 정답 이미지
    "과장 국내 출장 일비 표": "p2",
    "해외 출장 일비 달러": "p3",
    "경조사 휴가 일수": "p4",
    "연차휴가": "p4",
    "경조금 화환": "p5",
    "교육비 지원 한도": "p5",
    "호텔 영수증": "receipt",
    "직급별 출장비 막대그래프": "chart",
}
queries = {q: g for q, g in queries.items() if g in images}      # SMOKE 모드에서는 있는 이미지만

def rank_images(qs, M=None, prompt_name="SearchQuery"):
    M = IMG if M is None else M
    Q = eg.encode(list(qs), prompt_name=prompt_name, normalize_embeddings=True)
    S = eg.similarity(Q, M).numpy()
    out = []
    for (q, gold), row in zip(qs.items(), S):
        o = np.argsort(-row)
        out.append({"질문": q, "정답": gold,
                    "1등": f"{img_names[o[0]]} {row[o[0]]:.3f}", "2등": f"{img_names[o[1]]} {row[o[1]]:.3f}",
                    "3등": f"{img_names[o[2]]} {row[o[2]]:.3f}", "맞음": img_names[o[0]] == gold})
    return pd.DataFrame(out).set_index("질문"), S

def thumb(ax, key, title, color):
    """이미지 한 장을 작은 칸에 그리고 테두리 색으로 표시한다."""
    ax.imshow(images[key]); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(3)
    ax.set_title(title, fontsize=8, color=color)

# 3-1. 질문 하나 — '연차휴가'를 이미지 8장과 비교해 점수가 높은 순으로 줄 세우기
Q1 = "연차휴가"
s1 = eg.similarity(eg.encode([Q1], prompt_name="SearchQuery", normalize_embeddings=True), IMG).numpy()[0]
order = np.argsort(-s1)
fig, axes = plt.subplots(1, len(order), figsize=(2.0 * len(order), 3.3))
for r, (ax, j) in enumerate(zip(axes, order), 1):
    ok = img_names[j] == queries[Q1]
    thumb(ax, img_names[j], f"{r}등 {img_names[j]}\n{s1[j]:.3f}" + ("  (정답)" if ok else ""), "green" if ok else "gray")
fig.suptitle(f"질문 '{Q1}' → 정답 이미지 {queries[Q1]} · 점수(코사인)가 높은 순", fontsize=11)
plt.tight_layout(rect=(0, 0, 1, 0.86)); plt.show()

💡 **해석** — '연차휴가'의 정답 p4(제4장 휴가: 연차·경조사 휴가 표)가 0.715로 1등입니다. **OCR을 한 글자도 하지 않고** 스캔 페이지 그림만으로 찾았습니다.
다만 점수 차가 작습니다. 1등과 2등(p5 0.666)의 차이는 0.049이고, 2등부터 8등(p6 0.620)까지도 0.046 안에 몰려 있습니다. 점수는 "얼마나 가까운가"의 **순서**로만 읽습니다.

### 3-2. 질문 8개 전체를 그림으로

In [ ]:
# 3-2. 질문 8개 전체 — 질문 | 정답 이미지 | 모델이 고른 1~3등
# 테두리: 검정 = 정답 이미지(기준), 초록 = 순위 안의 정답 이미지, 빨강 = 정답이 아닌 이미지
tbl, S_img = rank_images(queries)
fig, axes = plt.subplots(len(queries), 5, figsize=(10, 2.3 * len(queries)))
for (q, gold), row, ax in zip(queries.items(), S_img, axes):
    ax[0].text(0.5, 0.5, f"질문\n'{q}'", ha="center", va="center", fontsize=10, wrap=True); ax[0].axis("off")
    thumb(ax[1], gold, f"정답: {gold}", "black")
    for r, j in enumerate(np.argsort(-row)[:3], 1):
        ok = img_names[j] == gold
        thumb(ax[r + 1], img_names[j], f"{r}등 {img_names[j]} {row[j]:.3f}", "green" if ok else "red")
plt.tight_layout(); plt.show()

💡 **해석** — 그림 한 줄이 질문 하나입니다. 왼쪽부터 질문, 미리 정한 정답 이미지(검은 테두리), 모델이 고른 1~3등입니다. 초록 테두리는 순위 안에서 정답 이미지가 놓인 자리, 빨강은 정답이 아닌 이미지입니다.
- **1등이 초록인 질문 5개**: 경조사 휴가 일수·연차휴가(p4), 경조금 화환(p5), 호텔 영수증(receipt), 직급별 출장비 막대그래프(chart)
- **1등이 빨강인 질문 3개**는 모두 정답이 **글자가 빽빽한 표 페이지**입니다. '과장 국내 출장 일비 표'와 '해외 출장 일비 달러'는 막대그래프 chart가 1등이었습니다. '교육비 지원 한도'는 영수증이 1등이었습니다.
- 틀린 3문항도 정답 이미지는 모두 2등(초록)이었습니다.

### 3-3. 표로 정리

In [ ]:
display(tbl)
print(f"hit@1 = {tbl['맞음'].mean():.2f}  ({tbl['맞음'].sum()}/{len(tbl)})")

fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.imshow(S_img, cmap="Blues"); ax.set_xticks(range(len(img_names)), img_names, rotation=30); ax.set_yticks(range(len(queries)), list(queries))
for i in range(S_img.shape[0]):
    for j in range(S_img.shape[1]):
        ax.text(j, i, f"{S_img[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if S_img[i, j] > S_img.max() - 0.04 else "black")
ax.set_title("질문 × 이미지 코사인 유사도"); plt.tight_layout(); plt.show()

💡 **해석** — 코랩 T4 실측입니다(강사 PC CPU에서도 순위가 같았고 점수 차는 0.005 이내).
- **hit@1 = 5/8**. OCR 없이 스캔 페이지에서 '연차휴가'·'경조금' 페이지를 찾았습니다. 영수증·차트처럼 **겉모습이 뚜렷한 이미지**는 1등과 2등의 차이도 큽니다(receipt 0.779 대 0.639, chart 0.764 대 0.647).
- 틀린 세 문항은 모두 표 페이지입니다. 280토큰으로 A4 한 쪽을 요약하면 작은 글자 정보가 뭉개집니다. '일비'가 든 질문 두 개는 직급별 국내 출장비 막대그래프가 1등이었습니다(해외 출장 질문인데도). 5절의 🟡에서 토큰을 늘려 봅니다.
- 점수 범위(0.6~0.8)가 좁습니다. 표 페이지가 정답인 질문 6개는 1등과 2등 차이가 0.012~0.051이라 **임계값으로 자르기 어렵습니다**.

## 4. 텍스트와 이미지를 한 공간에서 🔵
09번의 텍스트 청크 13개도 같은 모델로 임베딩해 **이미지 8장과 한 인덱스(21개)** 에 넣습니다. 질문 하나에 대해 이미지와 텍스트가 섞인 순위가 나옵니다.
3절처럼 먼저 질문 두 개의 섞인 순위를 그림으로 보고(4-1), 질문 8개 전체를 표로 정리합니다(4-2).
- 파란 테두리 = 이미지, 주황 테두리 = 텍스트 청크(앞부분 글자를 보여 줌)
- **(정답)** = 정답 이미지, 또는 질문과 같은 내용을 담은 텍스트 청크(예: '과장 국내 출장 일비 표' → 표 1 청크 `s2_t1_domestic`)

### 4-1. 질문 두 개 — 섞인 순위를 그림으로

In [ ]:
import textwrap
chunks = load_json("hanbit_chunks_gt.json")
chunk_text = {c["id"]: c["text"] for c in chunks}
TXT = eg.encode([c["text"] for c in chunks], prompt_name="Document", normalize_embeddings=True)
mix_keys = [("이미지", k) for k in img_names] + [("텍스트", c["id"]) for c in chunks]
MIX = np.vstack([IMG, TXT])
print("한 인덱스:", MIX.shape, f"← 이미지 {len(img_names)}개 + 텍스트 청크 {len(chunks)}개")

text_gold = {   # 질문과 같은 내용을 담은 텍스트 청크 (막대그래프는 표 1과 같은 숫자를 그림, 영수증은 같은 내용의 청크가 없다)
    "과장 국내 출장 일비 표": "s2_t1_domestic", "해외 출장 일비 달러": "s3_t2_overseas",
    "경조사 휴가 일수": "s4_t4_family_leave", "연차휴가": "s4_t3_annual_leave",
    "경조금 화환": "s5_t5_condolence", "교육비 지원 한도": "s5_t6_education",
    "직급별 출장비 막대그래프": "s2_t1_domestic"}

def mixed_rank(q):
    """질문 하나 → [((종류, 이름), 점수), ...] 점수가 높은 순"""
    s = eg.similarity(eg.encode([q], prompt_name="SearchQuery", normalize_embeddings=True), MIX).numpy()[0]
    return [(mix_keys[j], float(s[j])) for j in np.argsort(-s)]

def show_mixed(q, top=5):
    gold = {("이미지", queries.get(q)), ("텍스트", text_gold.get(q))}
    fig, axes = plt.subplots(1, top + 1, figsize=(2.3 * (top + 1), 3.4))
    axes[0].text(0.5, 0.5, f"질문\n'{q}'", ha="center", va="center", fontsize=11, wrap=True); axes[0].axis("off")
    for r, (ax, ((kind, key), sc)) in enumerate(zip(axes[1:], mixed_rank(q)[:top]), 1):
        title = f"{r}등 [{kind}] {key}\n{sc:.3f}" + ("  (정답)" if (kind, key) in gold else "")
        if kind == "이미지":
            thumb(ax, key, title, "tab:blue")
        else:
            ax.text(0.04, 0.96, textwrap.fill(chunk_text[key][:96], 16), va="top", fontsize=7, transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor("tab:orange"); sp.set_linewidth(3)
            ax.set_title(title, fontsize=8, color="tab:orange")
    plt.tight_layout(); plt.show()

# 4-1. 질문 두 개 — 이미지 8장 + 텍스트 13개가 섞인 순위의 1~5등
for q in ["과장 국내 출장 일비 표", "호텔 영수증"]:
    if q in queries:
        show_mixed(q)

💡 **해석** — 코랩 T4 실측입니다.
- **'과장 국내 출장 일비 표'**: 1등은 같은 내용의 텍스트 청크 `s2_t1_domestic`(0.830, 정답)입니다. 2·3·5등도 텍스트(제2장 본문 0.782, 표 2 해외 출장비 0.778, 표 0 개정 이력 0.754)이고, 이미지는 4등 chart(0.760) 하나뿐입니다. 정답 이미지 p2는 5등 안에 없습니다(4-2 표: 21개 중 10등).
- **'호텔 영수증'**: 1등은 영수증 이미지(0.779, 정답)이고, 2~5등은 모두 텍스트 청크(0.655~0.687)입니다. 규정집 글에는 영수증의 내용은 없고 '영수증'이라는 단어만 나옵니다(2등 `s2_text`의 '영수증을…'). 그래서 이미지가 1등입니다.
- 같은 인덱스에 섞어도 **어느 쪽이 위로 오는지는 질문에 따라 다릅니다**. 질문 8개 전체는 4-2 표에서 봅니다.

### 4-2. 질문 8개 전체를 표로

In [ ]:
# 4-2. 질문 8개 — 섞은 순위의 1등은 이미지일까 텍스트일까?
rows = []
for q, g in queries.items():
    rk = mixed_rank(q)
    (k1, n1), sc1 = rk[0]
    img_best = next((n, s) for (k, n), s in rk if k == "이미지")
    txt_best = next((n, s) for (k, n), s in rk if k == "텍스트")
    rows.append({"질문": q, "섞은 순위 1등": f"[{k1}] {n1} {sc1:.3f}",
                 "이미지 중 1등": f"{img_best[0]} {img_best[1]:.3f}", "텍스트 중 1등": f"{txt_best[0]} {txt_best[1]:.3f}",
                 "정답 이미지": g, f"정답 이미지 순위({len(mix_keys)}개 중)": next(i for i, (kn, _) in enumerate(rk, 1) if kn == ("이미지", g))})
display(pd.DataFrame(rows).set_index("질문"))

S_all = eg.similarity(eg.encode(list(queries), prompt_name="SearchQuery", normalize_embeddings=True), MIX).numpy()
print(f"평균 유사도 — 질문↔이미지 {S_all[:, :len(img_names)].mean():.3f} · 질문↔텍스트 청크 {S_all[:, len(img_names):].mean():.3f}")

💡 **해석** — 코랩 T4 실측입니다.
- 규정 내용을 묻는 질문 6개는 섞은 순위 1등이 모두 **텍스트 청크**였고, 모두 질문과 같은 내용의 청크(표 1~6 청크)였습니다. 정답 이미지는 21개 중 4~10등으로 밀렸습니다. '연차휴가'의 p4는 이미지 중에서는 1등(0.715)인데도 섞은 순위에서는 4등입니다.
- 영수증·막대그래프 질문 2개는 이미지가 1등입니다. 영수증은 텍스트 청크에 없는 내용이라 차이가 큽니다(0.779 대 0.687). 막대그래프는 표 1 청크(`s2_t1_domestic`)와 같은 숫자를 그린 것이라 차이가 0.010뿐입니다(0.764 대 0.754).
- 평균 유사도도 질문↔텍스트 청크(0.678)가 질문↔이미지(0.636)보다 높습니다. 텍스트 청크가 모두 규정 글이라 주제가 겹치는 영향도 섞여 있지만, 같은 내용끼리 비교해도 텍스트가 높습니다('연차휴가': 표 3 청크 0.819, p4 이미지 0.715). 같은 공간이라도 모달리티마다 점수대가 다른 이 현상을 **모달리티 격차(modality gap)** 라고 합니다.
- 그래서 실무에서는 이미지 인덱스와 텍스트 인덱스를 **따로 검색해 각각 상위 몇 개씩** 가져오거나, RRF(이론2 p78)처럼 **순위로** 합칩니다. 점수를 그대로 섞으면 한쪽(여기서는 텍스트)이 위를 차지하기 쉽습니다(질문 8개 중 6개에서 텍스트가 1등, 정답 이미지는 4~10등).

### 09번 텍스트 검색과 비교하면
- 09번에서 EG2는 텍스트 청크로 규정 질문 hit@1 0.85, hit@3 1.00이었습니다. 이 노트북의 이미지 검색은 기본 설정에서 hit@1 5/8입니다. **같은 모델이라도 텍스트로 뽑아 둔 쪽이 더 정확합니다.**
- 대신 이미지 검색은 OCR·파싱 단계가 없습니다. 파싱이 실패하는 문서(손글씨, 도장, 복잡한 도면, 사진)나 **이미지 자체가 답**인 경우(영수증, 차트)에 강합니다.
- 문서 페이지 검색에 특화된 모델(ColPali 계열, Qwen3-VL-Embedding)은 이 격차를 줄입니다. 오늘은 설치가 간단하고 게이트가 없는 EG2만 씁니다(이론2 p71).

## 5. 바꿔서 실행 🟡
1. **영어로 물으면?** 같은 내용을 영어로 물어 한국어 질문과 비교하세요.

In [ ]:
queries_en = {
    "domestic business trip daily allowance table": "p2",
    "family event leave days": "p4",
    "hotel receipt": "receipt",
    "bar chart of allowance by job grade": "chart",
}
queries_en = {q: g for q, g in queries_en.items() if g in images}
tbl_en, _ = rank_images(queries_en)
display(tbl_en)

💡 **해석** — 강사 환경(기본 280토큰)에서 영어 질문 4개 중 3개가 맞았습니다(p4 0.635, receipt 0.710, chart 0.691). "domestic business trip daily allowance table"은 한국어 질문과 똑같이 chart(0.680)가 1등, 정답 p2(0.648)는 3등이었습니다.
영어 질문의 1등 점수는 한국어 질문보다 0.06~0.08 낮았지만 **순위는 거의 같습니다**. 한국어 페이지를 영어로 찾을 수 있고, 한국어 질문이라서 불리하지도 않았습니다.

2. **시각 토큰을 늘리면?** `max_soft_tokens`는 이미지 한 장을 몇 개의 토큰으로 요약할지 정합니다(70~1120, 기본 280). 글자가 빽빽한 페이지에 1120을 줘 봅니다. 시간은 늘어납니다.

In [ ]:
def embed_images(max_soft_tokens):
    t0 = time.time()
    M = np.stack([eg.encode({"image": images[k]}, normalize_embeddings=True,
                            processing_kwargs={"image": {"max_soft_tokens": max_soft_tokens}}) for k in img_names])
    return M, time.time() - t0

rows = []
for mst in [280, 1120]:        # ← 70 이나 560 도 넣어 보세요
    M, sec = embed_images(mst)
    t, _ = rank_images(queries, M=M)
    rows.append({"max_soft_tokens": mst, "이미지 전체 임베딩(초)": round(sec, 1), "hit@1": f"{t['맞음'].sum()}/{len(t)}",
                 "틀린 질문": ", ".join(t.index[~t["맞음"]]) or "-"})
display(pd.DataFrame(rows).set_index("max_soft_tokens"))

💡 **해석** — 강사 환경(이미지 8장, CPU) 측정값:

| max_soft_tokens | 8장 임베딩 시간(CPU 한가할 때) | (CPU 바쁠 때) | hit@1 |
|---|---|---|---|
| 280 (기본) | 3.3초 | 15.5초 | 5/8 |
| 1120 | 20.2초 | 177초 | **8/8** |

- 토큰을 4배로 늘리자 틀렸던 표 페이지 세 개(p2·p3·p5)를 모두 찾았습니다. 작은 글자가 많은 문서 페이지에는 **해상도(토큰 수)가 곧 품질**입니다.
- 대가는 시간입니다. CPU에서 6~11배 느려졌습니다. 코랩 T4(bfloat16) 실측은 280토큰 5.4~5.8초 · 1120토큰 55~58초였고, hit@1은 5/8 → 7/8(과장 국내 출장 일비 표만 여전히 틀림)이었습니다(2026-10-18, 세 번 실행). 페이지가 수만 장이면 이 차이가 서버비가 됩니다.
- 다만 p2와 chart의 점수 차는 0.744 대 0.743으로 아슬아슬했습니다. 8개 질문으로 내린 결론이라는 점을 기억하세요.

### 🤔 생각해보기

**멀티모달 검색은 OCR+텍스트 검색을 대체할까, 보완할까?** 오늘 결과(정확도·속도·모달리티 격차)를 근거로 말해 보세요.

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- 오늘 결과만 보면 **보완**입니다. 같은 EG2라도 텍스트 청크 검색(09번 hit@1 0.85)이 이미지 페이지 검색(기본 5/8)보다 정확했습니다. 표 안의 숫자를 답해야 하는 RAG라면 결국 글자가 필요합니다. 페이지 이미지를 찾아도 LLM이 답하려면 OCR이나 VLM(07번)이 한 번 더 필요합니다.
- 반대로 **이미지 자체가 답인 경우**(영수증·차트·사진·도면)나 파싱이 깨지는 문서에서는 멀티모달 검색이 유일한 방법일 수 있습니다.
- 현실적인 설계는 **두 인덱스를 함께 두는 것**입니다. 텍스트 인덱스(파싱 결과)와 이미지 인덱스(페이지 이미지)를 따로 검색해 순위로 합칩니다. 페이지가 많고 파싱 품질이 나쁘면 문서 특화 멀티모달 모델(ColPali 계열)을 검토합니다.
- 비용도 따집니다. 시각 토큰을 늘리면 정확도는 오르지만 임베딩 시간이 몇 배로 늘어납니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 멀티모달 임베딩은 이미지와 텍스트를 한 벡터 공간에 넣는다. OCR 없이 한국어 질문으로 스캔 페이지를 찾을 수 있다.
- 이미지에는 접두어를 붙이지 않는다. 질문 텍스트에만 `prompt_name`을 붙인다.
- 글자가 빽빽한 페이지는 시각 토큰(`max_soft_tokens`)을 늘려야 잘 찾는다. 대신 느려진다.
- 모달리티마다 점수대가 다르다. 텍스트와 이미지를 섞을 때는 점수가 아니라 순위로 합친다.

> 다음 노트북: **11_mini_rag_local_vs_api — 근거를 대고 답하는 도우미 v1**